<div dir="rtl" align="right">

# קורס כריה וניתוח נתונים מתקדם

**קבוצה מספר:** 14  
**קישור ל-GitHub של הפרויקט:** [https://github.com/AlexTsiris/IMDB-project](https://github.com/AlexTsiris/IMDB-project)

### מגישים:

| שם הסטודנט | תעודת זהות |
| :--- | :--- |
| אלכסנדר ציריס | 11111111 |
| יואב ביג׳אוי | 22222222 |

</div>

---

<div dir="rtl" align="right"> 
קישור לGITHUB

In [67]:
#https://github.com/AlexTsiris/IMDB-project

<div dir="rtl" align="right"> 
    טעינת קובץ "title.basics.tsv"

In [3]:
import pandas as pd

In [4]:
df_basics = pd.read_csv('title.basics.tsv', sep='\t', low_memory=False, na_values='\\N')
df_basics.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894.0,NaN,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892.0,NaN,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892.0,NaN,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892.0,NaN,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893.0,NaN,1,Short


In [5]:
df_basics.columns

Index(['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult',
       'startYear', 'endYear', 'runtimeMinutes', 'genres'],
      dtype='object')

In [6]:
df_basics.isna().sum()

tconst                   0
titleType                0
primaryTitle            24
originalTitle           24
isAdult                  0
startYear          1465360
endYear           12320825
runtimeMinutes     7995029
genres              536180
dtype: int64

In [7]:
df_basics.shape

(12477189, 9)

<div dir="rtl" align="right">
סינון לפי movies

In [9]:
df_movies = df_basics[df_basics['titleType'] == 'movie'].copy()

In [10]:
#בידקה שהסינון עבד
df_movies["titleType"].unique()

array(['movie'], dtype=object)

<div dir="rtl" align="right">
סינון סרטים שמתחילים ב,N, O, n, o
לפי הנחיה

In [12]:
mask_n_o = df_movies['primaryTitle'].str.startswith(('N', 'O', 'n', 'o'), na=False)

<div dir="rtl" align="right">
בגלל שראינו שעמודות startYear           וגם runtimeMinutes      כוללים הרבה רשומות ריקות, עלינו לטפל בהם

In [14]:
df_movies = df_movies.dropna(subset=["startYear", "runtimeMinutes"])

In [15]:
#בדיקה שהטיפול בערכים חסרים עבד
df_movies[["startYear", "runtimeMinutes"]].isna().sum()

startYear         0
runtimeMinutes    0
dtype: int64

In [16]:
df_movies[["startYear", "runtimeMinutes"]].dtypes

startYear         float64
runtimeMinutes     object
dtype: object

<div dir="rtl" align="right">
ראינו שיש בעיה עם סוג של נתונים,צריכים לטפל

In [18]:
df_movies['startYear'] = df_movies['startYear'].astype(int)
df_movies['runtimeMinutes'] = df_movies['runtimeMinutes'].astype(int)

In [19]:
#בדיקה
df_movies[["startYear", "runtimeMinutes"]].dtypes

startYear         int32
runtimeMinutes    int32
dtype: object

In [20]:
df_filtered = df_movies[
    (df_movies['startYear'] <= 2024) & 
    (df_movies['runtimeMinutes'] >= 60) & 
    (df_movies['runtimeMinutes'] <= 300) & mask_n_o
]

C:\Users\Alex\AppData\Local\Temp\ipykernel_17580\4162992466.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_filtered = df_movies[


In [21]:
df_filtered.shape

(19860, 9)

<div dir="rtl" align="right">
טעינת קובץ "title.ratings.tsv"

In [23]:
df_ratings = pd.read_csv('title.ratings.tsv', sep='\t', low_memory=False)

In [24]:
df_ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2213
1,tt0000002,5.5,317
2,tt0000003,6.4,2327
3,tt0000004,5.1,199
4,tt0000005,6.2,3047


In [25]:
df_ratings.shape

(1668313, 3)

<div dir="rtl" align="right">
ניתן לראות שעמודה tconst, היא משותפת ל2 טבלאות נבצע מיזוג שמאלי LEFT JOIN 

In [27]:
df_merged = pd.merge(df_filtered, df_ratings, on='tconst', how='left')

In [28]:
#בודקים שכמות שורות נשארה זהה
df_merged.shape

(19860, 11)

In [29]:
#בודקים שעמודות חדשות נוספו
df_merged.columns

Index(['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult',
       'startYear', 'endYear', 'runtimeMinutes', 'genres', 'averageRating',
       'numVotes'],
      dtype='object')

<div dir="rtl" align="right">
טוענים קובץ "title.pribcipals.tsv"

In [31]:
df_principals = pd.read_csv('title.principals.tsv', sep='\t', usecols=['tconst', 'nconst', 'category'], low_memory=False)

In [32]:
df_principals.head()

,tconst,nconst,category
0,tt0000001,nm1588970,self
1,tt0000001,nm0005690,director
2,tt0000001,nm0005690,producer
3,tt0000001,nm0374658,cinematographer
4,tt0000002,nm0721526,director


In [33]:
df_principals.shape

(99253001, 3)

In [34]:
df_principals.isna().sum()

tconst      0
nconst      0
category    0
dtype: int64

<div dir="rtl" align="right">
מסננים דאטה רק לפי שחקנים

In [36]:
df_principals['category'].unique()

array(['self', 'director', 'producer', 'cinematographer', 'composer',
       'writer', 'editor', 'actor', 'actress', 'production_designer',
       'archive_footage', 'casting_director', 'archive_sound'],
      dtype=object)

In [37]:
df_actors = df_principals[df_principals['category'] == 'actor']

In [38]:
#מאשירים עמודות שצריכים
df_actors_only = df_actors[['tconst', 'nconst']]

In [39]:
#עושים גופ באי לפי סרט
grouped = df_actors_only.groupby('tconst')

In [40]:
#עמודה nconst תכלול רשימות של שחקנים 
actors_lists = grouped.agg(list)

In [41]:
actors_lists.head()

,nconst
tconst,
tt0000005,"[nm0443482, nm0653042]"
tt0000007,"[nm0179163, nm0183947]"
tt0000008,[nm0653028]
tt0000009,"[nm0183823, nm1309758]"
tt0000011,[nm3692297]


In [42]:
actors_lists = actors_lists.reset_index()

In [43]:
actors_lists.head()

,tconst,nconst
0,tt0000005,"[nm0443482, nm0653042]"
1,tt0000007,"[nm0179163, nm0183947]"
2,tt0000008,[nm0653028]
3,tt0000009,"[nm0183823, nm1309758]"
4,tt0000011,[nm3692297]


In [44]:
#נגדיר פונקציה שמקבלת עמודה של שחקנים ומחזירה עד 5 בכל רשימה
def get_top_5(actors):
    return actors[:5]

In [45]:
actors_lists['lead_actors_ids'] = actors_lists['nconst'].apply(get_top_5)

In [46]:
df_final_imdb = pd.merge(df_merged, actors_lists, on='tconst', how='left')

In [47]:
df_final_imdb.shape

(19860, 13)

In [48]:
columns_to_save = ['tconst', 'primaryTitle', 'startYear', 'genres', 'lead_actors_ids', 'runtimeMinutes', 'averageRating', 'numVotes']

In [49]:
df_final_imdb = df_final_imdb[columns_to_save]

In [50]:
df_final_imdb.columns

Index(['tconst', 'primaryTitle', 'startYear', 'genres', 'lead_actors_ids',
       'runtimeMinutes', 'averageRating', 'numVotes'],
      dtype='object')

In [51]:
df_final_imdb.dtypes

tconst              object
primaryTitle        object
startYear            int32
genres              object
lead_actors_ids     object
runtimeMinutes       int32
averageRating      float64
numVotes           float64
dtype: object

In [52]:
#טיפול בסוג נתונים של numVotes
df_final_imdb['numVotes'] = df_final_imdb['numVotes'].fillna(0).astype(int)
df_final_imdb['numVotes'].dtypes

dtype('int32')

In [53]:
df_final_imdb['genres']

0                  Drama,History
1                        Fantasy
2                Documentary,War
3        Adventure,Drama,Romance
4                            NaN
                  ...           
19855                        NaN
19856                      Drama
19857                      Drama
19858                     Comedy
19859                Documentary
Name: genres, Length: 19860, dtype: object

<div dir="rtl" align="right">
שלב WEB SCRABING 

In [55]:
# import requests
# from bs4 import BeautifulSoup
# import pandas as pd
# import time
# from tqdm import tqdm

# headers = {
#     "User-Agent": "ArielUniversity_StudentProject/1.0 (alex_tsiris@example.com)"
# }

# df_to_scrape = df_final_imdb.copy()

# # רשימה ריקה שתשמור את המילונים עם הנתונים שנאסוף לכל סרט
# all_wiki_data = []

# print(f"מתחיל איסוף נתונים לכל {len(df_to_scrape)} הסרטים...")

# # מעבר על כל שורה בטבלה עם סרגל התקדמות (tqdm)
# for index, row in tqdm(df_to_scrape.iterrows(), total=len(df_to_scrape)):
#     title = row['primaryTitle']
    
#     # יצירת קישור לחיפוש בוויקיפדיה
#     search_query = str(title).replace(" ", "+")
#     url = f"https://en.wikipedia.org/w/index.php?search={search_query}"
    
#     # אתחול מילון לאחסון הנתונים של הסרט הנוכחי
#     movie_data = {
#         'tconst': row['tconst'], 
#         'Language': None,
#         'Country': None,
#         'budget': None,
#         'BoxOffice': None,
#         'plot': None
#     }
    
#     try:
#         # שליחת בקשת GET לאתר
#         response = requests.get(url, headers=headers, timeout=10)
        
#         # בדיקה שהעמוד נטען בהצלחה
#         if response.status_code == 200:
#             soup = BeautifulSoup(response.text, 'html.parser')
            
#             # --- חילוץ תקציר העלילה (Plot) ---
#             paragraphs = soup.find_all('p')
#             if len(paragraphs) > 2:
#                 movie_data['plot'] = paragraphs[1].text.strip() + " " + paragraphs[2].text.strip()
            
#             # --- חילוץ נתונים מתיבת המידע (Infobox) ---
#             table = soup.find('table', class_="infobox")
#             if table:
#                 for tr in table.find_all('tr'):
#                     th = tr.find('th')
#                     td = tr.find('td')
                    
#                     if th and td:
#                         header_text = th.text.strip().lower()
#                         value_text = td.text.strip()
                        
#                         if 'language' in header_text:
#                             movie_data['Language'] = value_text
#                         elif 'countr' in header_text: 
#                             movie_data['Country'] = value_text
#                         elif 'budget' in header_text:
#                             movie_data['budget'] = value_text
#                         elif 'box office' in header_text:
#                             movie_data['BoxOffice'] = value_text
                            
#     except Exception as e:
#         # במקרה של שגיאה (למשל ניתוק רשת), נדלג לסרט הבא בלי לעצור את כל התוכנית
#         pass
    
#     # הוספת נתוני הסרט הנוכחי לרשימה הכוללת
#     all_wiki_data.append(movie_data)
    
#     # 🌟 מנגנון שמירת ביניים (גיבוי) - שומר קובץ כל 100 סרטים
#     if len(all_wiki_data) % 100 == 0:
#         pd.DataFrame(all_wiki_data).to_csv('wiki_backup_all.csv', index=False)
    
#     # השהייה קצרה של חצי שנייה כדי לא להעמיס על השרתים ולהימנע מחסימה (קריטי כשמריצים על 20 אלף רשומות)
#     time.sleep(0.5)

# # המרת רשימת המילונים ל-DataFrame של Pandas
# df_wiki_results = pd.DataFrame(all_wiki_data)

# # מיזוג הנתונים החדשים עם הנתונים המקוריים מ-IMDb
# df_final_all = pd.merge(df_to_scrape, df_wiki_results, on='tconst', how='left')

# # שמירת התוצאה הסופית לקובץ CSV
# df_final_all.to_csv('final_all_movies_dataset.csv', index=False)

In [73]:
import re
import numpy as np

# 1. טעינת נתוני ויקיפדיה מהגיבוי ומיזוג עם נתוני IMDb
try:
    df_wiki_backup = pd.read_csv('wiki_backup_all.csv')
    df_final = pd.merge(df_final_imdb, df_wiki_backup, on='tconst', how='left')
    print(f"מיזוג הושלם: {len(df_final)} שורות סה\"כ.")
except FileNotFoundError:
    print("שגיאה: קובץ הגיבוי 'wiki_backup_all.csv' לא נמצא בתיקייה.")

# 2. פונקציה לניקוי נתוני כסף (תקציב והכנסות) והמרה למיליונים
def clean_money(text):
    if pd.isna(text):
        return np.nan
    
    text = str(text).lower()
    text = re.sub(r'\[.*?\]', '', text) # הסרת הערות שוליים
    
    # חילוץ מספרים. אם יש כמה (טווח), נחשב ממוצע בהמשך
    raw_nums = re.findall(r'[\d\.]+', text.replace(',', ''))
    
    clean_nums = []
    for n in raw_nums:
        if n == '.': continue
        # תיקון למספרים עם נקודות במקום פסיקים (למשל 900.000.000)
        if n.count('.') > 1:
            n = n.replace('.', '')
        clean_nums.append(float(n))
    
    if not clean_nums:
        return np.nan
    
    # חישוב ממוצע במידה ויש טווח
    result = sum(clean_nums) / len(clean_nums)
    
    # המרה לפורמט של מיליונים
    if 'billion' in text:
        result *= 1000
    elif 'million' not in text and result > 1000:
        result /= 1000000
        
    return round(result, 2)

# הפעלת הניקוי על העמודות הרלוונטיות
df_final['budget'] = df_final['budget'].apply(clean_money)
df_final['BoxOffice'] = df_final['BoxOffice'].apply(clean_money)

# 3. עיבוד עמודת ז'אנרים לרשימות (List of Strings)
df_final['genres'] = df_final['genres'].apply(lambda x: x.split(',') if pd.notna(x) else [])

# 4. שמירה לקובץ סופי
df_final.to_csv('dataset.csv', index=False)
print("\nהקובץ dataset.csv נשמר בהצלחה.")

# תצוגה של תחילת הדאטה
df_final.head()

מיזוג הושלם: 19860 שורות סה"כ.

הקובץ dataset.csv נשמר בהצלחה.


,tconst,primaryTitle,startYear,genres,lead_actors_ids,runtimeMinutes,averageRating,numVotes,Language,Country,budget,BoxOffice,plot
0,tt0003241,One Hundred Years of Mormonism,1913,"[Drama, History]",[nm0949546],90,5.0,25,NaN,United States,NaN,NaN,One Hundred Years of Mormonism is a 1913 film ...
1,tt0004391,Neptune's Daughter,1914,[Fantasy],"[nm0790137, nm0920607, nm0107543, nm0607864, n...",70,5.5,72,NaN,NaN,NaN,NaN,NaN
2,tt0005831,On the Firing Line with the Germans,1915,"[Documentary, War]",NaN,90,6.8,30,NaN,NaN,NaN,NaN,NaN
3,tt0005832,On the Night Stage,1915,"[Adventure, Drama, Romance]","[nm0249187, nm0562197, nm0366586, nm0154184, n...",62,5.7,96,SilentEnglish intertitles,United States,NaN,NaN,On the Night Stage is a 1915 American silent ...
4,tt0007121,Neutraal Nederland,1917,[],NaN,165,NaN,0,NaN,NaN,NaN,NaN,NaN


In [75]:
# 1. חישוב כמות הערכים החסרים (NaN) בכל עמודה
missing_count = df_final.isna().sum()

# 2. חישוב אחוז הערכים החסרים
missing_percent = (df_final.isna().sum() / len(df_final)) * 100

# 3. קבלת סוג הנתונים (Data Type) של כל עמודה
data_types = df_final.dtypes

# 4. איסוף הנתונים לטבלה אחת מסודרת עבור הדוח
missing_data_summary = pd.DataFrame({
    'סוג נתון': data_types,
    'כמות ערכים חסרים': missing_count,
    'אחוז ערכים חסרים (%)': missing_percent.round(2)
})

# 5. הצגת התוצאה על המסך
display(missing_data_summary)

,סוג נתון,כמות ערכים חסרים,אחוז ערכים חסרים (%)
tconst,object,0,0.00
primaryTitle,object,0,0.00
startYear,int32,0,0.00
genres,object,0,0.00
lead_actors_ids,object,3593,18.09
runtimeMinutes,int32,0,0.00
averageRating,float64,6174,31.09
numVotes,int32,0,0.00
Language,object,17204,86.63
Country,object,17265,86.93
